# MODEL 1: LOGISTIC REGRESSION
# Bank Marketing Dataset

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


In [4]:
# 1. load Dataset
df = pd.read_csv("C:/Users/ASUS/Documents/bank-full.csv", sep=";")
print(df.head)
print(df.info())
print("Shape:", df.shape)
print("Columns:", df.columns)

<bound method NDFrame.head of        age           job   marital  education default  balance housing loan  \
0       58    management   married   tertiary      no     2143     yes   no   
1       44    technician    single  secondary      no       29     yes   no   
2       33  entrepreneur   married  secondary      no        2     yes  yes   
3       47   blue-collar   married    unknown      no     1506     yes   no   
4       33       unknown    single    unknown      no        1      no   no   
...    ...           ...       ...        ...     ...      ...     ...  ...   
45206   51    technician   married   tertiary      no      825      no   no   
45207   71       retired  divorced    primary      no     1729      no   no   
45208   72       retired   married  secondary      no     5715      no   no   
45209   57   blue-collar   married  secondary      no      668      no   no   
45210   37  entrepreneur   married  secondary      no     2971      no   no   

         contact  day

In [5]:
# 2. BASIC DATA CHECKS

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nTarget distribution:")
print(df["y"].value_counts())

print("\nTarget distribution (%):")
print(df["y"].value_counts(normalize=True) * 100)


Missing values:
0

Duplicate rows:
0

Target distribution:
y
no     39922
yes     5289
Name: count, dtype: int64

Target distribution (%):
y
no     88.30152
yes    11.69848
Name: proportion, dtype: float64


In [6]:
# 3. ENCODE TARGET VARIABLE


df["y"] = df["y"].map({
    "no": 0,
    "yes": 1
})

print("\nEncoded target distribution:")
print(df["y"].value_counts())


Encoded target distribution:
y
0    39922
1     5289
Name: count, dtype: int64


In [7]:
# 4. REMOVE TARGET AND POTENTIAL DATA LEAKAGE


X = df.drop(columns=["y", "duration"])
y = df["y"]

print("\nFeatures used:")
print(X.columns.tolist())


Features used:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']


In [8]:
# 5. DEFINE FEATURES

numeric_features = [
    "age",
    "balance",
    "day",
    "campaign",
    "pdays",
    "previous"
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome"
]


In [9]:
# 6. PREPROCESSING PIPELINE

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [10]:
# 7. LOGISTIC REGRESSION MODEL


logistic_regression = LogisticRegression(
    C=1.0,
    solver="liblinear",
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)


In [11]:
# 8. COMPLETE MODEL PIPELINE

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", logistic_regression)
    ]
)


In [15]:
# 9. TRAIN-TEST SPLIT


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("\nTraining set:", X_train.shape)
print("Testing set:", X_test.shape)


Training set: (31647, 15)
Testing set: (13564, 15)


In [16]:
# 10. TRAIN MODEL

print("\nTraining Logistic Regression model...")

model.fit(X_train, y_train)

print("Training completed.")


Training Logistic Regression model...
Training completed.


In [17]:
# 11. GENERATE PREDICTIONS


y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

In [18]:
# 12. CALCULATE PERFORMANCE METRICS

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

cm = confusion_matrix(y_test, y_pred)

In [19]:
# 13. DISPLAY RESULTS


print("\n" + "=" * 60)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 60)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "No Subscription",
            "Subscription"
        ],
        zero_division=0
    )
)


LOGISTIC REGRESSION RESULTS
Accuracy:  0.7564
Precision: 0.2682
Recall:    0.6257
F1 Score:  0.3754
ROC-AUC:   0.7701

Confusion Matrix:
[[9267 2710]
 [ 594  993]]

Classification Report:
                 precision    recall  f1-score   support

No Subscription       0.94      0.77      0.85     11977
   Subscription       0.27      0.63      0.38      1587

       accuracy                           0.76     13564
      macro avg       0.60      0.70      0.61     13564
   weighted avg       0.86      0.76      0.79     13564

